In [1]:
from pathlib import Path
from scipy.io import loadmat
import sys
import os

# Robust path finding for data.mat
current_path = Path.cwd()
possible_data_paths = [
    current_path / 'data' / 'data.mat',
    current_path.parent / 'data' / 'data.mat',
    current_path.parent.parent / 'data' / 'data.mat',
    # Fallback absolute path
    Path('/home/luky/skola/KalmanNet-for-state-estimation/data/data.mat')
]

dataset_path = None
for p in possible_data_paths:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None or not dataset_path.exists():
    print("Warning: data.mat not found automatically.")
    dataset_path = Path('data/data.mat')

print(f"Dataset path: {dataset_path}")

# Add project root to sys.path (2 levels up from debug/test)
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"Project root added: {project_root}")

mat_data = loadmat(dataset_path)
print(mat_data.keys())


Dataset path: /home/luky/skola/KalmanNet-main/data/data.mat
Project root added: /home/luky/skola/KalmanNet-main
dict_keys(['__header__', '__version__', '__globals__', 'hB', 'souradniceGNSS', 'souradniceX', 'souradniceY', 'souradniceZ'])


In [2]:
import torch
import matplotlib.pyplot as plt
from utils import trainer
from utils import utils
from Systems import DynamicSystem
import Filters
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from scipy.io import loadmat
from scipy.interpolate import RegularGridInterpolator
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [3]:
mat_data = loadmat(dataset_path)

souradniceX_mapa = mat_data['souradniceX']
souradniceY_mapa = mat_data['souradniceY']
souradniceZ_mapa = mat_data['souradniceZ']
souradniceGNSS = mat_data['souradniceGNSS'] 
x_axis_unique = souradniceX_mapa[0, :]
y_axis_unique = souradniceY_mapa[:, 0]

print(f"Dimensions of 1D X axis: {x_axis_unique.shape}")
print(f"Dimensions of 1D Y axis: {y_axis_unique.shape}")
print(f"Dimensions of 2D elevation data Z: {souradniceZ_mapa.shape}")

terMap_interpolator = RegularGridInterpolator(
    (y_axis_unique, x_axis_unique),
    souradniceZ_mapa,
    bounds_error=False, 
    fill_value=np.nan
)

def terMap(px, py):
    # Query bilinear interpolation over the terrain map
    points_to_query = np.column_stack((py, px))
    return terMap_interpolator(points_to_query)

Dimensions of 1D X axis: (2500,)
Dimensions of 1D Y axis: (2500,)
Dimensions of 2D elevation data Z: (2500, 2500)


In [4]:
import torch
from Systems import DynamicSystemTAN

state_dim = 4
obs_dim = 3
dT = 1
q = 1

F = torch.tensor([[1.0, 0.0, dT, 0.0],
                   [0.0, 1.0, 0.0, dT],
                   [0.0, 0.0, 1.0, 0.0],
                   [0.0, 0.0, 0.0, 1.0]])

Q = q* torch.tensor([[dT**3/3, 0.0, dT**2/2, 0.0],
                   [0.0, dT**3/3, 0.0, dT**2/2],
                   [dT**2/2, 0.0, dT, 0.0],
                   [0.0, dT**2/2, 0.0, dT]])
R = torch.tensor([[3.0**2, 0.0, 0.0],
                   [0.0, 1.0**2, 0.0],
                   [0.0, 0.0, 1.0**2]])

initial_velocity_np = souradniceGNSS[:2, 1] - souradniceGNSS[:2, 0]
# initial_velocity_np = torch.from_numpy()
initial_velocity = torch.from_numpy(np.array([0,0]))

initial_position = torch.from_numpy(souradniceGNSS[:2, 0])
x_0 = torch.cat([
    initial_position,
    initial_velocity
]).float()
print(x_0)

P_0 = torch.tensor([[25.0, 0.0, 0.0, 0.0],
                    [0.0, 25.0, 0.0, 0.0],
                    [0.0, 0.0, 0.5, 0.0],
                    [0.0, 0.0, 0.0, 0.5]])
import torch.nn.functional as func

def h_nl_differentiable(x: torch.Tensor, map_tensor, x_min, x_max, y_min, y_max) -> torch.Tensor:
    batch_size = x.shape[0]

    px = x[:, 0]
    py = x[:, 1]

    px_norm = 2.0 * (px - x_min) / (x_max - x_min) - 1.0
    py_norm = 2.0 * (py - y_min) / (y_max - y_min) - 1.0

    sampling_grid = torch.stack((px_norm, py_norm), dim=1).view(batch_size, 1, 1, 2)

    vyska_terenu_batch = func.grid_sample(
        map_tensor.expand(batch_size, -1, -1, -1),
        sampling_grid, 
        mode='bilinear', 
        padding_mode='border',
        align_corners=True
    )

    vyska_terenu = vyska_terenu_batch.view(batch_size)

    
    vx_w = x[:, 2]
    vy_w = x[:, 3]

    result = torch.stack([vyska_terenu, vx_w, vy_w], dim=1)
    
    return result

x_axis_unique = souradniceX_mapa[0, :]
y_axis_unique = souradniceY_mapa[:, 0]
terMap_tensor = torch.from_numpy(souradniceZ_mapa).float().unsqueeze(0).unsqueeze(0).to(device)
x_min, x_max = x_axis_unique.min(), x_axis_unique.max()
y_min, y_max = y_axis_unique.min(), y_axis_unique.max()

h_wrapper = lambda x: h_nl_differentiable(
    x, 
    map_tensor=terMap_tensor, 
    x_min=x_min, 
    x_max=x_max, 
    y_min=y_min, 
    y_max=y_max
)

system_model = DynamicSystemTAN(
    state_dim=state_dim,
    obs_dim=obs_dim,
    Q=Q.float(),
    R=R.float(),
    Ex0=x_0.float(),
    P0=P_0.float(),
    F=F.float(),
    h=h_wrapper,
    x_axis_unique=x_axis_unique, 
    y_axis_unique=y_axis_unique,
    device=device
)

tensor([1487547.1250, 6395520.5000,       0.0000,       0.0000])
INFO: DynamicSystemTAN inicializován s hranicemi mapy:
  X: [1476611.42, 1489541.47]
  Y: [6384032.63, 6400441.34]


In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from utils import utils
import torch
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import random
from copy import deepcopy
from state_NN_models import TAN
from utils import trainer 

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import os
from utils import trainer

class NavigationDataManager:

    def __init__(self, data_dir):
        self.data_dir = data_dir
        
    def get_dataloader(self, seq_len, split='train', shuffle=True, batch_size=32):
        path = os.path.join(self.data_dir, f'len_{seq_len}', f'{split}.pt')
        
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Dataset nenalezen: {path}")
            
        data = torch.load(path)
        x = data['x'] #  [Batch, Seq, DimX]
        y = data['y'] #  [Batch, Seq, DimY] - RAW DATA
        
        dataset = TensorDataset(x, y)
        
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

DATA_DIR = './generated_data_synthetic_controlled-extended-trajectories'

data_manager = NavigationDataManager(DATA_DIR)

curriculum_schedule = [
    {
        'phase_id': 1,
        'seq_len': 10,          
        'batch_size': 256
    },
    {
        'phase_id': 2,
        'seq_len': 100,           
        'batch_size': 128
    },
    {
        'phase_id': 3,
        'seq_len': 300,                    
        'batch_size': 64
    },
        {
        'phase_id': 4,
        'seq_len': 500,                   
        'batch_size': 64  
    }
]

datasets_cache = {} 

for phase in curriculum_schedule:
    seq_len = phase['seq_len']
    bs = phase['batch_size']
    
    
    try:
        train_loader = data_manager.get_dataloader(seq_len=seq_len, split='train', shuffle=True, batch_size=bs)
        val_loader = data_manager.get_dataloader(seq_len=seq_len, split='val', shuffle=False, batch_size=bs)
        
        datasets_cache[phase['phase_id']] = (train_loader, val_loader)

        
    except FileNotFoundError as e:
        print(f"   ⚠️ Error: {e}")


In [7]:
import torch
import torch.nn.functional as F
import numpy as np
from copy import deepcopy
from utils import losses

def training_session_trajectory_with_gaussian_nll_training_fcn(
    model, train_loader, val_loader, device,
    total_train_iter, learning_rate, clip_grad,
    J_samples, validation_period, logging_period,
    warmup_iterations=0, store_model_based_on_hybrid_score=False, 
    calibration_parameter=1.0, lambda_mse=1.0, 
    target_seq_len=None
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    best_val_anees = float('inf')
    best_hybrid_score = float('inf')
    score_at_best = {"val_nll": 0.0, "val_mse": 0.0, "hybrid_score": None}
    best_iter_count = 0
    best_model_state = None
    train_iter_count = 0
    done = False
    state_dim = model.system_model.state_dim
    
    while not done:
        model.train()
        for x_true_batch, y_meas_batch in train_loader:
            if train_iter_count >= total_train_iter: done = True; break
            
            x_true_batch = x_true_batch.to(device)
            y_meas_batch = y_meas_batch.to(device)
            
            if target_seq_len is not None:
                current_len = min(x_true_batch.shape[1], target_seq_len)
                x_true_batch = x_true_batch[:, :current_len, :]
                y_meas_batch = y_meas_batch[:, :current_len, :]
            
            optimizer.zero_grad()
            batch_size, seq_len, _ = x_true_batch.shape
            
            effective_batch_size = batch_size * J_samples
            
            x_true_expanded = x_true_batch.repeat(J_samples, 1, 1)
            y_meas_expanded = y_meas_batch.repeat(J_samples, 1, 1)
            
            model.reset(batch_size=effective_batch_size, initial_state=x_true_expanded[:, 0, :])
            
            current_trajectory_x_hats = []
            current_trajectory_regs = []
            
            for t in range(1, seq_len):
                y_t = y_meas_expanded[:, t, :]
                x_filtered_t, reg_t = model.step(y_t)
                current_trajectory_x_hats.append(x_filtered_t)
                current_trajectory_regs.append(reg_t)
                
            stacked_x_hats = torch.stack(current_trajectory_x_hats, dim=1)
            ensemble_trajectories = stacked_x_hats.view(J_samples, batch_size, seq_len - 1, state_dim)
            
            x_hat_sequence = ensemble_trajectories.mean(dim=0)  
            cov_diag_sequence = ensemble_trajectories.var(dim=0)
            
            regularization_loss = torch.stack(current_trajectory_regs).sum() / J_samples
            target_sequence = x_true_batch[:, 1:, :]
            
            with torch.no_grad():
                current_train_mse = F.mse_loss(x_hat_sequence, target_sequence).item()
                mean_var = cov_diag_sequence.mean().item()
                max_var = cov_diag_sequence.max().item()
                min_var = cov_diag_sequence.min().item()

            if train_iter_count < warmup_iterations:
                mse_loss_val = F.mse_loss(x_hat_sequence, target_sequence)
                loss = mse_loss_val + regularization_loss
                nll_loss = torch.tensor(0.0)
                phase_name = "WARMUP (MSE)"
                anchor_penalty = mse_loss_val.item() 
            else:
                nll_loss = losses.gaussian_nll_safe(target_sequence, x_hat_sequence, cov_diag_sequence, min_var=1.0e-5, max_error_cap=100.0)
                mse_anchor = F.mse_loss(x_hat_sequence, target_sequence)
                
                loss = nll_loss + (lambda_mse * mse_anchor) + regularization_loss
                
                phase_name = f"NLL + Anchor(L={lambda_mse})"
                anchor_penalty = (lambda_mse * mse_anchor).item()
            
            if torch.isnan(loss): print("Collapse detected (NaN loss)"); done = True; break
            
            loss.backward()
            if clip_grad > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_iter_count += 1
            
            if train_iter_count % logging_period == 0:
                with torch.no_grad():
                    p1 = torch.sigmoid(model.dnn.concrete_dropout1.p_logit).item() if hasattr(model.dnn, 'concrete_dropout1') else 0.0
                    p2 = torch.sigmoid(model.dnn.concrete_dropout2.p_logit).item() if hasattr(model.dnn, 'concrete_dropout2') else 0.0
                    
                print(f"--- Iteration [{train_iter_count}/{total_train_iter}] | Phase: {phase_name} ---")
                print(f"    - Total Loss: {loss.item():.4f} (NLL: {nll_loss.item():.4f}, Anchor: {anchor_penalty:.4f}, Reg: {regularization_loss.item():.4f})")
                print(f"    - Train MSE:  {current_train_mse:.4f}")
                print(f"    - Variance:   Mean: {mean_var:.4f} | Min: {min_var:.4f} | Max: {max_var:.4f}")
                print(f"    - Dropout:    p1={p1:.3f}, p2={p2:.3f}")

            # --- Validation step ---
            if train_iter_count > 0 and train_iter_count % validation_period == 0:
                print(f"\n" + "="*50)
                print(f" VALIDATION AT ITERATION {train_iter_count}")
                print("="*50)
                model.eval()
                val_mse_list = []
                all_val_x_true_cpu, all_val_x_hat_cpu, all_val_P_hat_cpu = [], [], []

                with torch.no_grad():
                    for x_true_val_batch, y_meas_val_batch in val_loader:
                        x_true_val_batch = x_true_val_batch.to(device)
                        y_meas_val_batch = y_meas_val_batch.to(device)
                        
                        if target_seq_len is not None:
                            val_current_len = min(x_true_val_batch.shape[1], target_seq_len)
                            x_true_val_batch = x_true_val_batch[:, :val_current_len, :]
                            y_meas_val_batch = y_meas_val_batch[:, :val_current_len, :]

                        val_batch_size, val_seq_len, _ = x_true_val_batch.shape
                        val_effective_batch_size = val_batch_size * J_samples
                        
                        x_val_expanded = x_true_val_batch.repeat(J_samples, 1, 1)
                        y_val_expanded = y_meas_val_batch.repeat(J_samples, 1, 1)
                        
                        model.reset(batch_size=val_effective_batch_size, initial_state=x_val_expanded[:, 0, :])
                        
                        val_current_x_hats = []
                        for t in range(1, val_seq_len):
                            y_t_val = y_val_expanded[:, t, :]
                            x_filtered_t, _ = model.step(y_t_val)
                            val_current_x_hats.append(x_filtered_t)
                            
                        stacked_val_x_hats = torch.stack(val_current_x_hats, dim=1)
                        val_ensemble = stacked_val_x_hats.view(J_samples, val_batch_size, val_seq_len - 1, state_dim)
                        val_preds_seq = val_ensemble.mean(dim=0)
                        
                        val_target_seq = x_true_val_batch[:, 1:, :]
                        val_mse_list.append(F.mse_loss(val_preds_seq, val_target_seq).item())
                        
                        initial_state_val = x_true_val_batch[:, 0, :].unsqueeze(1)
                        full_x_hat = torch.cat([initial_state_val, val_preds_seq], dim=1)
                        
                        diff = val_ensemble - val_preds_seq.unsqueeze(0)
                        outer_prods = diff.unsqueeze(-1) @ diff.unsqueeze(-2)
                        val_covs_full = outer_prods.mean(dim=0)

                        P0 = model.system_model.P0.unsqueeze(0).repeat(val_batch_size, 1, 1).unsqueeze(1)
                        full_P_hat = torch.cat([P0, val_covs_full], dim=1)
                        
                        all_val_x_true_cpu.append(x_true_val_batch.cpu())
                        all_val_x_hat_cpu.append(full_x_hat.cpu())
                        all_val_P_hat_cpu.append(full_P_hat.cpu())

                avg_val_mse = np.mean(val_mse_list)
                final_x_true_list = torch.cat(all_val_x_true_cpu, dim=0)
                final_x_hat_list = torch.cat(all_val_x_hat_cpu, dim=0)
                final_P_hat_list = torch.cat(all_val_P_hat_cpu, dim=0)
                
                import utils.trainer as trainer_module
                avg_val_anees = trainer_module.calculate_anees_vectorized(final_x_true_list, final_x_hat_list, final_P_hat_list)
                
                print(f"  > Target ANEES: {state_dim:.2f}")
                print(f"  > Actual ANEES: {avg_val_anees:.4f}")
                print(f"  > Average MSE:  {avg_val_mse:.4f}")
                
                if not np.isnan(avg_val_anees) and avg_val_anees > 0:
                    if store_model_based_on_hybrid_score:
                        anees_penalty = abs(avg_val_anees - state_dim)
                        scaled_penalty = calibration_parameter * anees_penalty
                        current_hybrid_score = avg_val_mse + scaled_penalty
                        
                        print(f"  > Hybrid Score: {current_hybrid_score:.4f} (MSE: {avg_val_mse:.4f} + Penalty: {scaled_penalty:.4f})")
                        
                        if current_hybrid_score < best_hybrid_score:
                            print(f"  🟢 NEW BEST HYBRID SCORE! ({current_hybrid_score:.4f} < {best_hybrid_score:.4f}) -> Saving model.")
                            best_hybrid_score = current_hybrid_score
                            best_val_anees = avg_val_anees
                            best_iter_count = train_iter_count
                            score_at_best['val_mse'] = avg_val_mse
                            score_at_best['hybrid_score'] = current_hybrid_score
                            best_model_state = deepcopy(model.state_dict())
                    else:
                        if avg_val_anees < best_val_anees:
                            print(f"  🟢 NEW BEST ANEES! ({avg_val_anees:.4f} < {best_val_anees:.4f}) -> Saving model.")
                            best_val_anees = avg_val_anees
                            best_iter_count = train_iter_count
                            score_at_best['val_mse'] = avg_val_mse
                            best_model_state = deepcopy(model.state_dict())
                            
                print("="*50 + "\n")
                model.train()

    print("\nTraining completed.")
    if best_model_state:
        if store_model_based_on_hybrid_score:
            print(f"Loading best model from iteration {best_iter_count} with Hybrid Score {best_hybrid_score:.4f} (MSE: {score_at_best['val_mse']:.4f}, ANEES: {best_val_anees:.4f})")
        else:
            print(f"Loading best model from iteration {best_iter_count} with ANEES {best_val_anees:.4f}")
        model.load_state_dict(best_model_state)
    else:
        print("No best model was saved; returning last state.")

    return {
        "best_val_anees": best_val_anees,
        "best_val_nll": score_at_best['val_nll'],
        "best_val_mse": score_at_best['val_mse'],
        "best_hybrid_score": score_at_best.get('hybrid_score', None),
        "best_iter": best_iter_count,
        "final_model": model
    }

In [ ]:
import torch
import copy
import os
import gc

# --- 1. CACHE MAPPING FOR AVAILABLE DATA ---
seq_len_to_idx = {
    10: 1,    
    100: 2,   
    300: 3,    
    500: 4     
}

# --- 2. FINAL SOTA CURRICULUM SCHEDULE ---
curriculum_schedule = [
    {
        'phase_id': 1, 'target_seq_len': 10, 'source_data_len': 10,
        'iters': 600, 'lr': 1e-3, 'clip_grad': 1.0,
        'mse_warmup_iters': 300, 'lambda_mse': 1.0,
        'calibration_parameter': 0.1, 'j_samples': 10  
    },
    # {
    #     'phase_id': 2, 'target_seq_len': 30, 'source_data_len': 100,
    #     'iters': 1000, 'lr': 1e-4, 'clip_grad': 0.5,
    #     'mse_warmup_iters': 300, 'lambda_mse': 2.0,
    #     'calibration_parameter': 0.25, 'j_samples': 20   
    # },
    # {
    #     'phase_id': 3, 'target_seq_len': 60, 'source_data_len': 100,
    #     'iters': 1000, 'lr': 5e-5, 'clip_grad': 0.2,
    #     'mse_warmup_iters': 300, 'lambda_mse': 5.0,
    #     'calibration_parameter': 0.25, 'j_samples': 20   
    # },
    {
        'phase_id': 4, 'target_seq_len': 100, 'source_data_len': 100,
        'iters': 1000, 'lr': 5e-5, 'clip_grad': 0.2,
        'mse_warmup_iters': 400, 'lambda_mse': 5.0,
        'calibration_parameter': 0.25, 'j_samples': 10   
    },
    {
        'phase_id': 5, 'target_seq_len': 300, 'source_data_len': 300,
        'iters': 1000, 'lr': 1e-5, 'clip_grad': 0.1,
        'mse_warmup_iters': 300, 'lambda_mse': 5.0,
        'calibration_parameter': 0.5, 'j_samples': 5  
    },
    {
        'phase_id': 6, 'target_seq_len': 500, 'source_data_len': 500,
        'iters': 1500, 'lr': 5e-6, 'clip_grad': 0.1,
        'mse_warmup_iters': 400, 'lambda_mse': 5.0,
        'calibration_parameter': 0.5, 'j_samples': 5  
    }
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# --- 3. MODEL INITIALIZATION ---
print("=== INITIALIZING BKN MODEL ===")
from state_NN_models import NCLT

state_knet2 = TAN.StateBayesianKalmanNetTAN(
        system_model=system_model, 
        device=device,
        hidden_size_multiplier=14,       
        output_layer_multiplier=4,
        num_gru_layers=1,
        init_max_dropout=0.4, 
        init_min_dropout=0.2
).to(device)

# --- 4. DIRECTORY SETUP & TRAINING LOGIC ---

# Create weights directory if it doesn't exist
weights_dir = "weights"
if not os.path.exists(weights_dir):
    os.makedirs(weights_dir)

# Create checkpoints directory inside weights
checkpoints_dir = os.path.join(weights_dir, "checkpoints")
if not os.path.exists(checkpoints_dir):
    os.makedirs(checkpoints_dir)

save_path = os.path.join(weights_dir, 'bkn_new_longer_data.pth')

if os.path.exists(save_path):
    print(f"\n✅ Found existing model at '{save_path}'.")
    print("   Skipping training and loading saved weights...")
    state_knet2.load_state_dict(torch.load(save_path, map_location=device))
    print("   Model is ready for evaluation.")
else:
    print(f"\n🚀 STARTING CURRICULUM TRAINING ({len(curriculum_schedule)} phases)")

    for phase in curriculum_schedule:
        # Clear CUDA cache before each phase
        torch.cuda.empty_cache()
        gc.collect()

        pid = phase['phase_id']
        target_len = phase['target_seq_len']
        source_len = phase['source_data_len']
        current_j_samples = phase.get('j_samples', 10)
        
        print(f"\n" + "="*60)
        print(f"🌊 PHASE {pid}: Target length {target_len} (Data source: {source_len})")
        print(f"   Parameters: LR={phase['lr']}, LambdaMSE={phase['lambda_mse']}, J_samples={current_j_samples}")
        print("="*60)
        
        try:
            cache_idx = seq_len_to_idx[source_len]
            train_loader = datasets_cache[cache_idx][0]
            val_loader = datasets_cache[cache_idx][1]
            print(f"   -> Loading loader from datasets_cache[{cache_idx}] (original data length: {source_len})")
        except KeyError:
            print(f"❌ ERROR: Index for length {source_len} not defined in 'seq_len_to_idx'!")
            break
        except IndexError:
            print(f"❌ ERROR: Index {cache_idx} does not exist in datasets_cache!")
            break

        # Run training session
        phase_results = training_session_trajectory_with_gaussian_nll_training_fcn(
            model=state_knet2,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            
            total_train_iter=phase['iters'],
            learning_rate=phase['lr'],
            warmup_iterations=phase['mse_warmup_iters'],
            clip_grad=phase['clip_grad'],
            store_model_based_on_hybrid_score=True,
            calibration_parameter=phase.get('calibration_parameter', 0.0),
            
            lambda_mse=phase.get('lambda_mse', 1.0),
            target_seq_len=target_len,
            
            J_samples=current_j_samples, 
            validation_period=10,
            logging_period=10,
        )
        
        # Save intermediate phase checkpoint
        phase_ckpt_path = os.path.join(checkpoints_dir, f"bkn_phase_{pid}_len{target_len}.pth")
        torch.save(state_knet2.state_dict(), phase_ckpt_path)
        print(f"✅ Phase {pid} completed. Model saved to {phase_ckpt_path}")

    print("\n🏆 FULL TRAINING COMPLETED.")
    
    # Save the final model
    torch.save(state_knet2.state_dict(), save_path)
    print(f"Final model saved to '{save_path}'.")

device: cuda
=== INITIALIZING BKN MODEL ===
initialized with gradient terrain

🚀 STARTING CURRICULUM TRAINING (4 phases)

🌊 PHASE 1: Target length 10 (Data source: 10)
   Parameters: LR=0.001, LambdaMSE=1.0, J_samples=10
   -> Loading loader from datasets_cache[1] (original data length: 10)
--- Iteration [10/600] | Phase: WARMUP (MSE) ---
    - Total Loss: 8.6793 (NLL: 0.0000, Anchor: 8.6774, Reg: 0.0019)
    - Train MSE:  8.6774
    - Variance:   Mean: 7.9441 | Min: 0.0072 | Max: 121.4028
    - Dropout:    p1=0.249, p2=0.298

 VALIDATION AT ITERATION 10
  > Target ANEES: 4.00
  > Actual ANEES: 15.3018
  > Average MSE:  7.8095
  > Hybrid Score: 8.9397 (MSE: 7.8095 + Penalty: 1.1302)
  🟢 NEW BEST HYBRID SCORE! (8.9397 < inf) -> Saving model.

--- Iteration [20/600] | Phase: WARMUP (MSE) ---
    - Total Loss: 6.0965 (NLL: 0.0000, Anchor: 6.0946, Reg: 0.0019)
    - Train MSE:  6.0946
    - Variance:   Mean: 6.6286 | Min: 0.0131 | Max: 101.2222
    - Dropout:    p1=0.251, p2=0.296

 VALIDA

In [ ]:
if False:
    # save model.
    save_path = f'bkn_new_longer_data.pth'
    torch.save(state_knet2.state_dict(), save_path)
    print(f"Model saved to '{save_path}'.")

# Synthetic test

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import Filters
import os
from tqdm import tqdm
from Filters import TAN

# === CONFIGURATION ===
TEST_DATA_PATH = './generated_data_synthetic_controlled-extended-trajectories/test_set/test.pt'
PLOT_PER_ITERATION = True    # Plot a graph for each trajectory?
MAX_TEST_SAMPLES = 20        # Number of trajectories to evaluate from the test set
J_EVALUATION = 50            # Number of Monte Carlo samples for BKN (Ensemble size)

print(f"=== BKN EVALUATION ON TEST SET (with ANEES) ===")
print(f"Loading data from: {TEST_DATA_PATH}")

# 1. Load Test Set
if not os.path.exists(TEST_DATA_PATH):
    raise FileNotFoundError(f"File {TEST_DATA_PATH} does not exist!")

# Assume 'device' is defined, fallback if not
if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_data = torch.load(TEST_DATA_PATH, map_location=device)
X_test_all = test_data['x']  # Ground Truth [N, Seq, 4]
Y_test_all = test_data['y']  # Measurements [N, Seq, 3]

n_samples = min(X_test_all.shape[0], MAX_TEST_SAMPLES)
print(f"Number of test trajectories: {n_samples}")
print(f"Ensemble size (BKN): {J_EVALUATION}")
print(f"Sequence length: {X_test_all.shape[1]}")
print("Models: BKN vs. UKF vs. PF vs. APF")

# 2. Initialization for data collection
detailed_results = []
trajectory_history = []
agg_mse = {"BKN": [], "UKF": [], "PF": [], "APF": []}
agg_pos = {"BKN": [], "UKF": [], "PF": [], "APF": []}
agg_anees = {"BKN": [], "UKF": [], "PF": [], "APF": []} # New list for ANEES

# Ensure BKN is in eval mode
state_knet2.eval() 

# --- HELPER FUNCTION FOR ANEES ---
def calculate_anees(gt, est, P):
    """
    Calculates the Average Normalized Estimation Error Squared (ANEES).
    gt: Ground Truth [T, Dim] (NumPy)
    est: Estimate [T, Dim] (NumPy)
    P: Covariance matrix [T, Dim, Dim] (NumPy)
    """
    T = min(len(gt), len(est), len(P))
    anees_vals = []
    
    # Truncate to the same length
    gt = gt[:T]
    est = est[:T]
    P = P[:T]
    
    for t in range(T):
        e_t = gt[t] - est[t] # Error at time t
        P_t = P[t]
        
        try:
            # Covariance inversion
            # Add a small epsilon to the diagonal for numerical stability if singular
            if np.linalg.cond(P_t) > 1e10:
                P_t = P_t + np.eye(P_t.shape[0]) * 1e-6
                
            P_inv = np.linalg.inv(P_t)
            
            # Mahalanobis distance: e^T * P^-1 * e
            anees_t = e_t.T @ P_inv @ e_t
            anees_vals.append(anees_t)
        except np.linalg.LinAlgError:
            anees_vals.append(np.nan)
            
    return np.nanmean(anees_vals)

# --- MAIN LOOP ---
for i in tqdm(range(n_samples), desc="Evaluating"):
    
    # A) Data preparation
    x_gt_tensor = X_test_all[i].to(device)
    y_obs_tensor = Y_test_all[i].to(device)
    
    x_gt = x_gt_tensor.cpu().numpy()
    seq_len = x_gt.shape[0]
    true_init_state = x_gt_tensor[0] 
    
    # --- B) BKN (Ensemble) ---
    with torch.no_grad():
        init_batch = true_init_state.unsqueeze(0).repeat(J_EVALUATION, 1)
        state_knet2.reset(batch_size=J_EVALUATION, initial_state=init_batch)
        
        bkn_preds = []
        y_input_batch = y_obs_tensor.unsqueeze(0).repeat(J_EVALUATION, 1, 1)
        
        for t in range(1, seq_len):
            y_t = y_input_batch[:, t, :]
            x_est, _ = state_knet2.step(y_t) 
            bkn_preds.append(x_est)
            
        if len(bkn_preds) > 0:
            bkn_preds_tensor = torch.stack(bkn_preds, dim=1) # [J, Seq-1, 4]
            full_bkn_ensemble = torch.cat([init_batch.unsqueeze(1), bkn_preds_tensor], dim=1) # [J, Seq, 4]
            
            # Mean Estimate
            x_est_mean = full_bkn_ensemble.mean(dim=0)
            x_est_bkn = x_est_mean.cpu().numpy()
            
            # --- BKN COVARIANCE CALCULATION ---
            # P = 1/(J-1) * sum((x_j - x_mean) * (x_j - x_mean)^T)
            # Centering
            residuals = full_bkn_ensemble - x_est_mean.unsqueeze(0) # [J, Seq, 4]
            # Permute for batch matmul: [Seq, J, 4] and [Seq, 4, J]
            residuals = residuals.permute(1, 2, 0) # [Seq, 4, J]
            
            # Batch matrix multiplication: (Seq, 4, J) @ (Seq, J, 4) -> (Seq, 4, 4)
            P_bkn_tensor = torch.bmm(residuals, residuals.transpose(1, 2)) / (J_EVALUATION - 1)
            P_bkn = P_bkn_tensor.cpu().numpy()
            
        else:
            x_est_bkn = x_gt
            P_bkn = np.eye(4)[np.newaxis, :, :].repeat(len(x_gt), axis=0)

    # --- C) Classical Filters ---
    
    # UKF
    ukf_ideal = Filters.UnscentedKalmanFilter(system_model)
    ukf_res = ukf_ideal.process_sequence(y_seq=y_obs_tensor, Ex0=true_init_state, P0=system_model.P0)
    x_est_ukf = ukf_res['x_filtered'].cpu().numpy()
    # Get P for UKF
    P_ukf = ukf_res.get('P_filtered', ukf_res.get('P', None))
    if P_ukf is not None: P_ukf = P_ukf.cpu().numpy()

    # PF
    pf = TAN.ParticleFilterTAN(system_model, num_particles=1000) 
    pf_res = pf.process_sequence(y_seq=y_obs_tensor, Ex0=true_init_state, P0=system_model.P0)
    x_est_pf = pf_res['x_filtered'].cpu().numpy()
    P_pf = pf_res.get('P_filtered', pf_res.get('P', None))
    if P_pf is not None: P_pf = P_pf.cpu().numpy()

    # APF
    apf = TAN.AuxiliaryParticleFilterTAN(system_model, num_particles=2000) 
    apf_res = apf.process_sequence(y_seq=y_obs_tensor, Ex0=true_init_state, P0=system_model.P0)
    x_est_apf = apf_res['x_filtered'].cpu().numpy()
    P_apf = apf_res.get('P_filtered', apf_res.get('P', None))
    if P_apf is not None: P_apf = P_apf.cpu().numpy()
    
    # --- D) Error and ANEES Calculation ---
    min_len = min(len(x_gt), len(x_est_bkn), len(x_est_ukf))
    
    def calc_metrics(est, gt, P_mat):
        diff = est[:min_len] - gt[:min_len]
        mse = np.mean(np.sum(diff[:, :2]**2, axis=1)) 
        pos_err = np.mean(np.sqrt(diff[:, 0]**2 + diff[:, 1]**2))
        
        anees = np.nan
        if P_mat is not None:
            anees = calculate_anees(gt[:min_len], est[:min_len], P_mat[:min_len])
            
        return mse, pos_err, anees

    # Calculate for all
    mse_bkn, pos_bkn, anees_bkn = calc_metrics(x_est_bkn, x_gt, P_bkn)
    mse_ukf, pos_ukf, anees_ukf = calc_metrics(x_est_ukf, x_gt, P_ukf)
    mse_pf, pos_pf, anees_pf = calc_metrics(x_est_pf, x_gt, P_pf)
    mse_apf, pos_apf, anees_apf = calc_metrics(x_est_apf, x_gt, P_apf)
    
    # Store results
    agg_mse["BKN"].append(mse_bkn); agg_pos["BKN"].append(pos_bkn); agg_anees["BKN"].append(anees_bkn)
    agg_mse["UKF"].append(mse_ukf); agg_pos["UKF"].append(pos_ukf); agg_anees["UKF"].append(anees_ukf)
    agg_mse["PF"].append(mse_pf);   agg_pos["PF"].append(pos_pf);   agg_anees["PF"].append(anees_pf)
    agg_mse["APF"].append(mse_apf); agg_pos["APF"].append(pos_apf); agg_anees["APF"].append(anees_apf)

    detailed_results.append({
        "Run_ID": i + 1,
        "BKN_PosErr": pos_bkn, "BKN_ANEES": anees_bkn,
        "UKF_PosErr": pos_ukf, "UKF_ANEES": anees_ukf,
        "PF_PosErr": pos_pf,   "PF_ANEES": anees_pf,
        "APF_PosErr": pos_apf, "APF_ANEES": anees_apf
    })
    
    trajectory_history.append({
        "gt": x_gt,
        "bkn_est": x_est_bkn,
        "bkn_P": P_bkn,
        "ukf_est": x_est_ukf,
        "ukf_P": P_ukf
    })
    
    # E) Plotting
    if PLOT_PER_ITERATION:
        fig = plt.figure(figsize=(12, 6))
        plt.plot(x_gt[:, 0], x_gt[:, 1], 'k-', linewidth=3, alpha=0.3, label='Ground Truth')
        plt.plot(x_est_bkn[:, 0], x_est_bkn[:, 1], 'g-', linewidth=2, label=f'BKN (Err: {pos_bkn:.1f}m, ANEES: {anees_bkn:.1f})')
        plt.plot(x_est_ukf[:, 0], x_est_ukf[:, 1], 'b--', linewidth=1, label=f'UKF (Err: {pos_ukf:.1f}m, ANEES: {anees_ukf:.1f})')
        # Uncomment below to plot PF/APF
        # plt.plot(x_est_pf[:, 0], x_est_pf[:, 1], 'r:', linewidth=1, alpha=0.6, label='PF')
        plt.title(f"Test Trajectory {i+1}")
        plt.xlabel("X [m]")
        plt.ylabel("Y [m]")
        plt.legend()
        plt.axis('equal')
        plt.grid(True)
        plt.show()

# --- PRINT RESULTS ---
df_results = pd.DataFrame(detailed_results)
print("\n" + "="*120)
print(f"DETAILED RESULTS (Position in meters | ANEES - ideal ~4.0)")
print("="*120)
pd.options.display.float_format = '{:,.2f}'.format
print(df_results[["Run_ID", "BKN_PosErr", "BKN_ANEES", "UKF_PosErr", "UKF_ANEES", "PF_PosErr", "APF_PosErr"]])

print("\n" + "="*120)
print(f"SUMMARY STATISTICS ({n_samples} trajectories)")
print("="*120)

def get_stats(key):
    return (np.nanmean(agg_mse[key]), np.nanstd(agg_mse[key]), 
            np.nanmean(agg_pos[key]), np.nanstd(agg_pos[key]),
            np.nanmean(agg_anees[key]), np.nanstd(agg_anees[key]))

bkn_s = get_stats("BKN")
ukf_s = get_stats("UKF")
pf_s = get_stats("PF")
apf_s = get_stats("APF")

# Format output table
header = f"{'Model':<10} | {'Pos Error [m] (Mean ± Std)':<30} | {'ANEES (Mean ± Std)':<30}"
print(header)
print("-" * len(header))
print(f"{'BKN':<10} | {bkn_s[2]:.2f} ± {bkn_s[3]:.2f} m {'':<14} | {bkn_s[4]:.2f} ± {bkn_s[5]:.2f}")
print(f"{'UKF':<10} | {ukf_s[2]:.2f} ± {ukf_s[3]:.2f} m {'':<14} | {ukf_s[4]:.2f} ± {ukf_s[5]:.2f}")
print(f"{'PF':<10} | {pf_s[2]:.2f} ± {pf_s[3]:.2f} m {'':<14} | {pf_s[4]:.2f} ± {pf_s[5]:.2f}")
print(f"{'APF':<10} | {apf_s[2]:.2f} ± {apf_s[3]:.2f} m {'':<14} | {apf_s[4]:.2f} ± {apf_s[5]:.2f}")
print("="*120)

# Graphical comparison (Boxplot Position Error)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.boxplot([agg_pos["BKN"], agg_pos["UKF"], agg_pos["PF"]], labels=['BKN', 'UKF', 'PF'], patch_artist=True, boxprops=dict(facecolor='lightblue'))
plt.title("Position Error [m]")
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

# Graphical comparison (Boxplot ANEES)
plt.subplot(1, 2, 2)
# Filter NaN for boxplot
anees_data = [
    [x for x in agg_anees["BKN"] if not np.isnan(x)],
    [x for x in agg_anees["UKF"] if not np.isnan(x)],
    [x for x in agg_anees["PF"] if not np.isnan(x)]
]
plt.boxplot(anees_data, labels=['BKN', 'UKF', 'PF'], patch_artist=True, boxprops=dict(facecolor='lightgreen'))
plt.axhline(y=4.0, color='r', linestyle='--', label='Ideal (4.0)')
plt.title("ANEES (Consistency)")
plt.legend()
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Check if data from the previous cell is stored
if 'trajectory_history' not in globals() or len(trajectory_history) < 2:
    print("⚠️ Error: 'trajectory_history' data is missing. Make sure you made the modification in the main loop.")
else:
    for traj_idx in range(len(trajectory_history)):
        data = trajectory_history[traj_idx]
        
        gt = data["gt"]
        bkn_est = data["bkn_est"]
        bkn_P = data["bkn_P"]
        ukf_est = data["ukf_est"]
        ukf_P = data["ukf_P"]
        
        seq_len = min(len(gt), len(bkn_est), len(ukf_est))
        time_steps = np.arange(seq_len)
        
        # Calculation of deviations (Error = Estimate - Ground Truth)
        err_bkn_x = bkn_est[:seq_len, 0] - gt[:seq_len, 0]
        err_bkn_y = bkn_est[:seq_len, 1] - gt[:seq_len, 1]
        
        err_ukf_x = ukf_est[:seq_len, 0] - gt[:seq_len, 0]
        err_ukf_y = ukf_est[:seq_len, 1] - gt[:seq_len, 1]
        
        # Calculation of 3-sigma bounds from covariance matrices
        # (index 0,0 is X variance; index 1,1 is Y variance)
        std_bkn_x = np.sqrt(bkn_P[:seq_len, 0, 0])
        std_bkn_y = np.sqrt(bkn_P[:seq_len, 1, 1])
        
        std_ukf_x = np.sqrt(ukf_P[:seq_len, 0, 0])
        std_ukf_y = np.sqrt(ukf_P[:seq_len, 1, 1])

        # --- PLOTTING ---
        fig, axs = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
        fig.suptitle(f'Trajectory {traj_idx + 1}: Consistency Analysis (Error vs. 3σ Bounds)', fontsize=16)

        # 1. BKN - Position X
        axs[0, 0].plot(time_steps, err_bkn_x, 'g-', linewidth=2, label='BKN Error X')
        axs[0, 0].fill_between(time_steps, -3*std_bkn_x, 3*std_bkn_x, color='green', alpha=0.2, label='BKN 3σ Bounds')
        axs[0, 0].axhline(0, color='black', linestyle='--', linewidth=1)
        axs[0, 0].set_title('Bayesian KalmanNet - Error in X-axis')
        axs[0, 0].set_ylabel('Error [m]')
        axs[0, 0].legend(loc='upper right')
        axs[0, 0].grid(True, linestyle=':', alpha=0.7)

        # 2. BKN - Position Y
        axs[1, 0].plot(time_steps, err_bkn_y, 'g-', linewidth=2, label='BKN Error Y')
        axs[1, 0].fill_between(time_steps, -3*std_bkn_y, 3*std_bkn_y, color='green', alpha=0.2, label='BKN 3σ Bounds')
        axs[1, 0].axhline(0, color='black', linestyle='--', linewidth=1)
        axs[1, 0].set_title('Bayesian KalmanNet - Error in Y-axis')
        axs[1, 0].set_xlabel('Time step (k)')
        axs[1, 0].set_ylabel('Error [m]')
        axs[1, 0].legend(loc='upper right')
        axs[1, 0].grid(True, linestyle=':', alpha=0.7)

        # 3. UKF - Position X
        axs[0, 1].plot(time_steps, err_ukf_x, 'b-', linewidth=2, label='UKF Error X')
        axs[0, 1].fill_between(time_steps, -3*std_ukf_x, 3*std_ukf_x, color='blue', alpha=0.2, label='UKF 3σ Bounds')
        axs[0, 1].axhline(0, color='black', linestyle='--', linewidth=1)
        axs[0, 1].set_title('Unscented Kalman Filter - Error in X-axis')
        axs[0, 1].legend(loc='upper right')
        axs[0, 1].grid(True, linestyle=':', alpha=0.7)

        # 4. UKF - Position Y
        axs[1, 1].plot(time_steps, err_ukf_y, 'b-', linewidth=2, label='UKF Error Y')
        axs[1, 1].fill_between(time_steps, -3*std_ukf_y, 3*std_ukf_y, color='blue', alpha=0.2, label='UKF 3σ Bounds')
        axs[1, 1].axhline(0, color='black', linestyle='--', linewidth=1)
        axs[1, 1].set_title('Unscented Kalman Filter - Error in Y-axis')
        axs[1, 1].set_xlabel('Time step (k)')
        axs[1, 1].legend(loc='upper right')
        axs[1, 1].grid(True, linestyle=':', alpha=0.7)

        # Aligning Y-axes for a fair comparison between BKN and UKF
        max_y_val_x = max(np.max(np.abs(err_bkn_x)), np.max(3*std_bkn_x), np.max(np.abs(err_ukf_x)), np.max(3*std_ukf_x)) * 1.1
        max_y_val_y = max(np.max(np.abs(err_bkn_y)), np.max(3*std_bkn_y), np.max(np.abs(err_ukf_y)), np.max(3*std_ukf_y)) * 1.1
        
        axs[0, 0].set_ylim(-max_y_val_x, max_y_val_x)
        axs[0, 1].set_ylim(-max_y_val_x, max_y_val_x)
        axs[1, 0].set_ylim(-max_y_val_y, max_y_val_y)
        axs[1, 1].set_ylim(-max_y_val_y, max_y_val_y)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()